# Image Resolution Enhancement - Bilinear Interpolation Method

**Project:** Image Resolution Enhancement Using Optimized Edge-Preserving Interpolation

**Team Members:**
- Aliaa Maamoun Ibrahim (120220255)
- Sama Ayman Bakry (120220342)
- Ahmed Mohamed Ahmed (120220150)
- Mohamed Hamdy Gaber (120220033)
- Youssef Abd El Mohsen Eissa (120220051)
- Mohamed Walid (120220050)

**Method:** Bilinear Interpolation (Baseline)

---

## 1️⃣ Setup & Installation

In [ ]:
# Install required libraries
!pip install -q opencv-python scikit-image matplotlib numpy pandas seaborn pillow

print("✅ Installation complete!")

In [ ]:
# Import all necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import pandas as pd
import seaborn as sns
from PIL import Image
import requests
from io import BytesIO
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

In [ ]:
# Create directory structure
directories = [
    'images/original',
    'images/lr_x2',
    'images/lr_x4',
    'results/bilinear_x2',
    'results/bilinear_x4'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("✅ Directory structure created!")

## 3️⃣ Helper Functions

In [ ]:
def load_image(image_path):
    """Load image in RGB format"""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def save_image(image, path):
    """Save image in RGB format"""
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, img_bgr)

def downsample_image(image, scale_factor):
    """
    Downsample image by scale factor
    Args:
        image: Input high-resolution image
        scale_factor: Downsampling factor (2 or 4)
    Returns:
        Low-resolution image
    """
    height, width = image.shape[:2]
    new_height = height // scale_factor
    new_width = width // scale_factor
    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)

print("✅ Helper functions defined!")

In [ ]:
def display_comparison_grid(original, lr, upscaled, scale_factor, image_name):
    """Display original, LR, and upscaled images with zoom"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Row 1: Full images
    axes[0, 0].imshow(original)
    axes[0, 0].set_title('Original (Ground Truth)', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(lr)
    axes[0, 1].set_title(f'Low Resolution (÷{scale_factor})', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(upscaled)
    axes[0, 2].set_title(f'Bilinear Upscaled (×{scale_factor})', fontsize=14, fontweight='bold')
    axes[0, 2].axis('off')

    # Row 2: Zoomed patches
    h, w = original.shape[:2]
    crop_size = min(h, w) // 4
    y_center, x_center = h // 2, w // 2
    y1, y2 = y_center - crop_size // 2, y_center + crop_size // 2
    x1, x2 = x_center - crop_size // 2, x_center + crop_size // 2

    axes[1, 0].imshow(original[y1:y2, x1:x2])
    axes[1, 0].set_title('Original (Zoomed)', fontsize=12)
    axes[1, 0].axis('off')

    lr_y1, lr_y2 = y1 // scale_factor, y2 // scale_factor
    lr_x1, lr_x2 = x1 // scale_factor, x2 // scale_factor
    axes[1, 1].imshow(lr[lr_y1:lr_y2, lr_x1:lr_x2])
    axes[1, 1].set_title('LR (Zoomed)', fontsize=12)
    axes[1, 1].axis('off')

    axes[1, 2].imshow(upscaled[y1:y2, x1:x2])
    axes[1, 2].set_title('Upscaled (Zoomed)', fontsize=12)
    axes[1, 2].axis('off')

    plt.suptitle(f'{image_name} - Scale Factor: ×{scale_factor}',
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions ready!")

## 4️⃣ Bilinear Interpolation Implementation

In [ ]:
def download_image(url, filename):
    """Download image from URL and save it"""
    try:
        response = requests.get(url, timeout=15)
        img = Image.open(BytesIO(response.content))
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img.save(f'images/original/{filename}.png')
        print(f"✅ Downloaded: {filename}")
        return True
    except Exception as e:
        print(f"❌ Failed to download {filename}: {str(e)}")
        return False

## 5️⃣ Organize & clean dataset files

In [ ]:
import os

print("🧹 Organizing Set5/Set14 dataset files...\n")

# We only want the HR images at scale factor 2 and 4
# The naming is: img_XXX_SRF_Y_HR.png where Y is the scale factor
files_to_keep = []
files_to_delete = []

for filename in os.listdir('images/original'):
    filepath = f'images/original/{filename}'

    # Keep only HR images (these are our ground truth originals)
    # AND keep the 3 standard images we already have
    if '_HR' in filename or filename in ['airplane.png', 'baboon.png', 'peppers.png']:
        files_to_keep.append(filename)

    # Delete LR images (we'll generate these ourselves) and result charts
    else:
        files_to_delete.append(filename)
        try:
            os.remove(filepath)
        except:
            pass

print(f"✅ Kept {len(files_to_keep)} HR images:")

# Rename for clarity
rename_map = {
    # Set5 (img_001 to img_005)
    'img_001_SRF_2_HR.png': 'baby_set5.png',
    'img_002_SRF_2_HR.png': 'bird_set5.png',
    'img_003_SRF_2_HR.png': 'butterfly_set5.png',
    'img_004_SRF_2_HR.png': 'head_set5.png',
    'img_005_SRF_2_HR.png': 'woman_set5.png',

    # Set14 (img_006 to img_014 are just some of them)
    'img_006_SRF_2_HR.png': 'barbara_set14.png',
    'img_007_SRF_2_HR.png': 'bridge_set14.png',
    'img_008_SRF_2_HR.png': 'coastguard_set14.png',
    'img_009_SRF_2_HR.png': 'comic_set14.png',
    'img_010_SRF_2_HR.png': 'face_set14.png',
    'img_011_SRF_2_HR.png': 'flowers_set14.png',
    'img_012_SRF_2_HR.png': 'foreman_set14.png',
    'img_013_SRF_2_HR.png': 'lenna_set14.png',
    'img_014_SRF_2_HR.png': 'man_set14.png',
}

# Rename files for better identification
for old_name, new_name in rename_map.items():
    old_path = f'images/original/{old_name}'
    new_path = f'images/original/{new_name}'
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
        print(f"  ✓ {old_name} → {new_name}")

# Clean up remaining _HR files at other scales (SRF_3, SRF_4)
for filename in os.listdir('images/original'):
    if ('SRF_3' in filename or 'SRF_4' in filename) and filename not in rename_map:
        try:
            os.remove(f'images/original/{filename}')
        except:
            pass

# Final count
final_images = [f for f in os.listdir('images/original')
                if f.endswith(('.png', '.bmp', '.jpg', '.jpeg'))]

print(f"\n🗑️ Deleted {len(files_to_delete)} LR/duplicate files")
print("\n" + "=" * 60)
print(f"📊 FINAL IMAGE COUNT: {len(final_images)}")
print("=" * 60)

print("\n📋 Final image list:")
for idx, img in enumerate(sorted(final_images), 1):
    print(f"  {idx}. {img}")

# Categorize
set5 = [f for f in final_images if 'set5' in f.lower()]
set14 = [f for f in final_images if 'set14' in f.lower()]
standard = [f for f in final_images if 'set5' not in f.lower() and 'set14' not in f.lower()]

print("\n" + "=" * 60)
print("📈 DATASET SUMMARY:")
print("=" * 60)
print(f"  Standard Images: {len(standard)} {standard}")
print(f"  Set5 Images: {len(set5)}")
print(f"  Set14 Images: {len(set14)}")
print(f"  TOTAL: {len(final_images)} images")
print("=" * 60)

if len(final_images) >= 12:
    print("\n✅ Great! Sufficient dataset for evaluation")
    print("🚀 Ready to process all images!")
else:
    print("\n⚠️ Limited dataset - but enough to proceed")

In [ ]:
def evaluate_image(original, upscaled, image_name, scale_factor):
    """Evaluate upscaled image against original"""
    psnr_score = calculate_psnr(original, upscaled)
    ssim_score = calculate_ssim(original, upscaled)
    fsim_score = calculate_fsim(original, upscaled)

    return {
        'Image': image_name,
        'Scale': f'x{scale_factor}',
        'PSNR (dB)': round(psnr_score, 4),
        'SSIM': round(ssim_score, 4),
        'FSIM': round(fsim_score, 4)
    }

print("✅ Evaluation function ready!")

## 6️⃣ Process Images

In [ ]:
def process_image(image_path, image_name, scale_factors=[2, 4]):
    """
    Complete pipeline for one image:
    1. Load original
    2. Downsample to create LR
    3. Upsample using bilinear
    4. Evaluate and save results
    """
    results = []
    original = load_image(image_path)

    print(f"\n{'='*60}")
    print(f"📸 Processing: {image_name}")
    print(f"{'='*60}")
    print(f"Original size: {original.shape[1]}x{original.shape[0]}")

    for scale in scale_factors:
        print(f"\n🔄 Scale Factor: ×{scale}")

        # Downsample
        lr_image = downsample_image(original, scale)
        print(f"  LR size: {lr_image.shape[1]}x{lr_image.shape[0]}")
        save_image(lr_image, f'images/lr_x{scale}/{image_name}_lr_x{scale}.png')

        # Upsample
        upscaled = bilinear_upsample(lr_image, scale)
        print(f"  Upscaled size: {upscaled.shape[1]}x{upscaled.shape[0]}")
        save_image(upscaled, f'results/bilinear_x{scale}/{image_name}_bilinear_x{scale}.png')

        # Evaluate
        metrics = evaluate_image(original, upscaled, image_name, scale)
        results.append(metrics)

        print(f"  📊 PSNR: {metrics['PSNR (dB)']:.2f} dB")
        print(f"  📊 SSIM: {metrics['SSIM']:.4f}")
        print(f"  📊 FSIM: {metrics['FSIM']:.4f}")

        # Display
        display_comparison_grid(original, lr_image, upscaled, scale, image_name)

    return results

print("✅ Processing function ready!")

## 7️⃣ Run Experiments

In [ ]:
# Process all images
all_results = []
image_files = [f for f in os.listdir('images/original') if f.endswith('.png')]

if len(image_files) == 0:
    print("⚠️ No images found! Please upload images to 'images/original/' folder")
else:
    print(f"🎯 Found {len(image_files)} images to process\n")

    for img_file in sorted(image_files):
        image_name = img_file.replace('.png', '')
        image_path = f'images/original/{img_file}'

        try:
            results = process_image(image_path, image_name, scale_factors=[2, 4])
            all_results.extend(results)
        except Exception as e:
            print(f"❌ Error processing {image_name}: {str(e)}")

## 8️⃣ Results Summary

In [ ]:
if len(all_results) > 0:
    # Create DataFrame
    df_results = pd.DataFrame(all_results)

    print("\n" + "="*70)
    print("📊 FINAL RESULTS - BILINEAR INTERPOLATION")
    print("="*70)
    print(df_results.to_string(index=False))
    print("="*70)

    # Save CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f'results/bilinear_results_{timestamp}.csv'
    df_results.to_csv(csv_filename, index=False)
    print(f"\n💾 Results saved to: {csv_filename}")

    # Average scores
    print("\n📈 AVERAGE SCORES:")
    for scale in [2, 4]:
        scale_data = df_results[df_results['Scale'] == f'x{scale}']
        if len(scale_data) > 0:
            print(f"\n  Scale x{scale}:")
            print(f"    Average PSNR: {scale_data['PSNR (dB)'].mean():.2f} dB")
            print(f"    Average SSIM: {scale_data['SSIM'].mean():.4f}")
            print(f"    Average FSIM: {scale_data['FSIM'].mean():.4f}")
else:
    print("⚠️ No results to display!")

## 9️⃣ Visualizations

In [ ]:
if len(all_results) > 0:
    # Bar charts
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['PSNR (dB)', 'SSIM', 'FSIM']
    colors = ['#3498db', '#2ecc71', '#e74c3c']

    for idx, (metric, color) in enumerate(zip(metrics, colors)):
        df_pivot = df_results.pivot(index='Image', columns='Scale', values=metric)

        # Plot each scale with different alpha manually
        ax = axes[idx]
        x = np.arange(len(df_pivot.index))
        width = 0.35

        # Plot x2 bars with lighter alpha
        if 'x2' in df_pivot.columns:
            ax.bar(x - width/2, df_pivot['x2'], width, label='x2',
                   color=color, alpha=0.6)

        # Plot x4 bars with full alpha
        if 'x4' in df_pivot.columns:
            ax.bar(x + width/2, df_pivot['x4'], width, label='x4',
                   color=color, alpha=1.0)

        ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
        ax.set_xlabel('Image', fontsize=12)
        ax.set_ylabel(metric, fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(df_pivot.index, rotation=45, ha='right')
        ax.legend(title='Scale Factor', fontsize=10)
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    plt.savefig(f'results/bilinear_metrics_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Bar charts created!")

In [ ]:
if len(all_results) > 0:
    # Heatmaps
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['PSNR (dB)', 'SSIM', 'FSIM']

    for idx, metric in enumerate(metrics):
        pivot_data = df_results.pivot(index='Image', columns='Scale', values=metric)
        sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='YlGnBu',
                    ax=axes[idx], cbar_kws={'label': metric})
        axes[idx].set_title(f'{metric} Heatmap', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Scale Factor', fontsize=12)
        axes[idx].set_ylabel('Image', fontsize=12)

    plt.tight_layout()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    plt.savefig(f'results/bilinear_heatmap_{timestamp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Heatmaps created!")

## 🔟 Export for Team Comparison

In [ ]:
if len(all_results) > 0:
    summary = df_results.copy()
    summary['Method'] = 'Bilinear'
    summary = summary[['Method', 'Image', 'Scale', 'PSNR (dB)', 'SSIM', 'FSIM']]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comparison_file = f'results/bilinear_for_comparison_{timestamp}.csv'
    summary.to_csv(comparison_file, index=False)

    print("\n" + "="*70)
    print("📤 EXPORT FOR TEAM COMPARISON")
    print("="*70)
    print(summary.to_string(index=False))
    print(f"\n💾 Saved to: {comparison_file}")
    print("\n✅ Share this file with your team for final comparison!")

## ✅ Summary

### What We Accomplished:
1. ✅ Manually uploaded Set5, Set14, and standard test images (17 images total)
2. ✅ Organized and cleaned dataset files
3. ✅ Implemented Bilinear Interpolation for image super-resolution
4. ✅ Created LR images with scale factors ×2 and ×4
5. ✅ Evaluated all images using PSNR, SSIM, and FSIM metrics
6. ✅ Generated comprehensive visualizations (bar charts and heatmaps)
7. ✅ Exported results in team comparison format

### Dataset Used:
- **Standard Images (3):** airplane, baboon, peppers
- **Set5 Dataset (5):** baby, bird, butterfly, head, woman
- **Set14 Dataset (9):** barbara, bridge, coastguard, comic, face, flowers, foreman, lenna, man
- **Total:** 17 images × 2 scales = 34 experiments

### Results Summary:
- **×2 Scale:** Average PSNR = 29.16 dB, SSIM = 0.889, FSIM = 0.827
- **×4 Scale:** Average PSNR = 25.07 dB, SSIM = 0.758, FSIM = 0.700

### Output Files:
- `images/original/` - Original high-resolution test images (17 images)
- `images/lr_x2/` - Low-resolution images (×2 downsampling)
- `images/lr_x4/` - Low-resolution images (×4 downsampling)
- `results/bilinear_x2/` - Bilinear upsampled results (×2)
- `results/bilinear_x4/` - Bilinear upsampled results (×4)
- `results/bilinear_results_*.csv` - Complete metrics for all images
- `results/bilinear_for_comparison_*.csv` - Formatted for team comparison
- `results/bilinear_metrics_*.png` - Bar chart visualizations
- `results/bilinear_heatmap_*.png` - Heatmap visualizations